In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

TARGET = f"{GOLD}.gold_supply_chain_kpi"

In [0]:
active_sites = (
    spark.table(f"{SILVER}.dim_site_scd2")
    .filter("is_current = true")
    .select("site_id")
    .distinct()
    .count()
)

active_patients = (
    spark.table(f"{SILVER}.dim_patient")
    .filter("patient_status = 'ACTIVE'")
    .count()
)

total_inventory = (
    spark.table(f"{SILVER}.fact_inventory")
    .agg(
        F.sum("available_stock")
    )
    .first()[0]
)

total_dispensed_quantity = (
    spark.table(f"{SILVER}.fact_dispensing")
    .agg(
        F.sum("quantity_dispensed")
    )
    .first()[0]
)

total_shipments = (
    spark.table(f"{SILVER}.fact_shipment_current")
    .count()
)

delayed_shipments = (
    spark.table(f"{SILVER}.fact_shipment_current")
    .filter("delivery_delay_days > 0")
    .count()
)

stockout_risk_sites = (
    spark.table(f"{GOLD}.gold_stockout_risk")
    .filter("stockout_risk_flag = 'HIGH_RISK'")
    .select("site_id")
    .distinct()
    .count()
)

In [0]:
from datetime import date

reporting_date = date.today()

kpi_data = [
    (
        reporting_date,
        active_sites,
        active_patients,
        total_inventory,
        total_dispensed_quantity,
        total_shipments,
        delayed_shipments,
        stockout_risk_sites
    )
]

kpi_df = spark.createDataFrame(
    kpi_data,
    [
        "reporting_date",
        "active_sites",
        "active_patients",
        "total_inventory",
        "total_dispensed_quantity",
        "total_shipments",
        "delayed_shipments",
        "stockout_risk_sites"
    ]
)

In [0]:
display(kpi_df)

In [0]:
(
    kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET)
)

In [0]:
display(
    spark.table(TARGET)
)

In [0]:
print(
    f"gold_supply_chain_kpi records : "
    f"{spark.table(TARGET).count()}"
)